In [1]:
import os
import pandas as pd
from asset_modeling.credit import loan_portfolio, private_credit_loan_model
from data.sofr import get_sofr_data

fred_api_key = os.getenv("FRED_API_KEY")

#### credit functions

##### private_credit_loan_model()
Models a single private credit loan and generates a quarterly payment schedule with amortization, interest, and fees.

**Arguments:**
- `investment_name`: Name/identifier for the investment
- `investment_date`: Date the investment/loan is initiated
- `maturity_date`: Date the loan matures
- `loan_size`: Principal amount of the loan
- `spread`: Spread in basis points
- `base_rate`: Base interest rate
- `sofr_assumption`: SOFR rate assumption
- `cash_interest_rate`: Annual cash interest rate (paid quarterly)
- `pik_interest`: Annual payment-in-kind interest rate (added to balance quarterly)
- `amortization`: Annual amortization rate applied to original loan amount
- `oid`: Original issue discount as a percentage of loan_size
- `exit_fee`: Fee as percentage of remaining balance at maturity/prepayment
- `prepayment_date`: Optional date when loan is prepaid (if None, uses maturity_date)

**Returns:** DataFrame with quarterly schedule including cash flows, balances, interest, and IRR

---

##### loan_portfolio()
Aggregates multiple loans into a portfolio summary by combining individual loan schedules and performing quarterly rollup.

**Arguments:**
- `schedule_of_investments`: DataFrame or dict with one row per loan containing all required columns for `private_credit_loan_model()`

**Returns:** DataFrame with quarterly portfolio totals (invested_amount, total_payment, remaining_balance_payment, ending_balance, beginning_balance)

---
#### Example: private_credit_loan_model() 

1) Determine Cash Rate
    - Total_Rate = SOFR + Spread = Cash Interest Rate + PIK Interest Rate
        - Example: 10 = 4 + 6 = 8 + 2
    Cash Interest Rate = SOFR + Spread - PIK Interest
        - Example: 8 = 4 + 6 - 2

2) Determine the beginning and ending balance by accounting for Amortization and PIK Interest
    Where:
        - Amortization (fixed on Par Value) = Loan Par Value * (Amortization Rate/4)
        - PIK intereest = Beginning Balance * (PIK Interest Rate/4)
        - Ending balance = Beginning Balance - Amortization + PIK Interest

3) Calculate Cash Interest
    - Ending Balance * (Cash Interest Rate/4)

4) Calculate End of Term Payments
    - Where:
        - payments are made in the last quarter when the term is over or the loan is prepaid
    - Exit Fee = Loan Par Value * Exit Fee
    - Balance Repayment = Beginning Balance at the last quarter date

5) Calculate Cash Flow
    - Where:
        - Interest and Amortization are paid quarterly
        - Balance Repayment and Exit Fees are paid in final quarter (Exit fee is based on par value)
    - Cash Payment = Interest + Amortization + Balance Repayment + Fees

In [2]:
### private_credit_loan_model()
# Example usage of the private_credit_loan_model function to create a single loan schedule

rates = get_sofr_data(api_key=fred_api_key, frequency='D')

loan = private_credit_loan_model(
    investment_name="Example Corp Term Loan",
    investment_date=pd.Timestamp("2024-12-31"),
    maturity_date=pd.Timestamp("2029-12-31"),
    par_value=1_000_000,
    spread=0.06,
    base_rate='SOFR',
    sofr_rates=rates,
    # sofr_assumption=0.04,
    # sofr_floor = 0.01
    pik_interest=0.02,
    amortization=0.01,
    oid=0.02,
    exit_fee=0.02,
    prepayment_date=None,
)

print(f"Loan schedule generated with {len(loan)} quarterly periods")
loan

Loan schedule generated with 21 quarterly periods


,investment_name,quarter_end,par_value,invested_amount,base_rate,sofr_rate,rate_status,spread,pik_rate,cash_interest_rate,...,prepayment_date,beginning_balance,amortization,pik_interest,ending_balance,cash_interest,fees,remaining_balance_payment,total_payment,irr
0,Example Corp Term Loan,2024-12-31,1000000,980000.0,SOFR,0.0449,actual,0.00,0.00,0.0000,...,None,1.000000e+06,0.0,0.000000,1.000000e+06,0.000000,0.0,0.000000e+00,-9.800000e+05,NaN
1,Example Corp Term Loan,2025-03-31,1000000,980000.0,SOFR,0.0441,actual,0.06,0.02,0.0841,...,None,1.000000e+06,2500.0,5000.000000,1.002500e+06,21025.000000,0.0,0.000000e+00,2.352500e+04,0.023488
2,Example Corp Term Loan,2025-06-30,1000000,980000.0,SOFR,0.0445,actual,0.06,0.02,0.0845,...,None,1.002500e+06,2500.0,5012.500000,1.005012e+06,21177.812500,0.0,0.000000e+00,2.367781e+04,0.024551
3,Example Corp Term Loan,2025-09-30,1000000,980000.0,SOFR,0.0424,actual,0.06,0.02,0.0824,...,None,1.005012e+06,2500.0,5025.062500,1.007538e+06,20703.257500,0.0,0.000000e+00,2.320326e+04,0.103596
4,Example Corp Term Loan,2025-12-31,1000000,980000.0,SOFR,0.0387,actual,0.06,0.02,0.0787,...,None,1.007538e+06,2500.0,5037.687813,1.010075e+06,19823.301542,0.0,0.000000e+00,2.232330e+04,0.103830
5,Example Corp Term Loan,2026-03-31,1000000,980000.0,SOFR,0.0368,actual,0.06,0.02,0.0768,...,None,1.010075e+06,2500.0,5050.376252,1.012626e+06,19393.444806,0.0,0.000000e+00,2.189344e+04,0.103648
6,Example Corp Term Loan,2026-06-30,1000000,980000.0,SOFR,0.0368,assumed,0.06,0.02,0.0768,...,None,1.012626e+06,2500.0,5063.128133,1.015189e+06,19442.412030,0.0,0.000000e+00,2.194241e+04,0.103518
7,Example Corp Term Loan,2026-09-30,1000000,980000.0,SOFR,0.0368,assumed,0.06,0.02,0.0768,...,None,1.015189e+06,2500.0,5075.943773,1.017765e+06,19491.624090,0.0,0.000000e+00,2.199162e+04,0.103421
8,Example Corp Term Loan,2026-12-31,1000000,980000.0,SOFR,0.0368,assumed,0.06,0.02,0.0768,...,None,1.017765e+06,2500.0,5088.823492,1.020354e+06,19541.082211,0.0,0.000000e+00,2.204108e+04,0.103345
9,Example Corp Term Loan,2027-03-31,1000000,980000.0,SOFR,0.0368,assumed,0.06,0.02,0.0768,...,None,1.020354e+06,2500.0,5101.767610,1.022955e+06,19590.787622,0.0,0.000000e+00,2.209079e+04,0.103285


In [3]:
schedule_of_investments = pd.DataFrame(
    {
        "investment_name": ["A Corp Term Loan", "B Corp Term Loan",  "C Corp Term Loan",  "D Corp Term Loan",  "E Corp Term Loan"],
        "investment_date": [pd.Timestamp("2020-03-31"), pd.Timestamp("2020-06-30"), pd.Timestamp("2020-09-30"), pd.Timestamp("2020-12-31"), pd.Timestamp("2021-03-31")],
        "maturity_date": [pd.Timestamp("2025-06-30"), pd.Timestamp("2027-06-30"), pd.Timestamp("2026-12-31"), pd.Timestamp("2028-03-31"), pd.Timestamp("2029-06-30")],
        "par_value": [1_000_000, 1_250_000, 1_000_000, 1_050_000, 1_600_000],
        "spread": [0.07, 0.06, 0.06, 0.08, 0.07],
        "base_rate": ['SOFR', 'SOFR', 'SOFR', 'SOFR', 'SOFR'],
        "sofr_assumption": ['actual', 'actual', 'actual', 'actual', 'actual'],
        "pik_interest": [0.02, 0.02, 0.02, 0.02, 0.02],
        "amortization": [0.01, 0.01, 0.01, 0.01, 0.01],
        "oid": [0.03, 0.02, 0.03, 0.04, 0.02],
        "exit_fee": [0.02, 0.02, 0.02, 0.02, 0.02],
        "prepayment_date": [None, None, None, None, None],
    }
)

portfolio, funds, funds_summary = loan_portfolio(schedule_of_investments, rates)

In [5]:
portfolio.head()

,quarter_end,invested_amount,total_payment,remaining_balance_payment,ending_balance,beginning_balance,irr,tvpi
0,2020-03-31,970000.0,-9.700000e+05,0.0,1.000000e+06,1.000000e+06,None,1.030928
1,2020-06-30,2195000.0,-1.209750e+06,0.0,2.252500e+06,2.250000e+06,0.075,1.033375
2,2020-09-30,3165000.0,-9.388932e+05,0.0,3.258138e+06,3.252500e+06,0.043699,1.044729
3,2020-12-31,4173000.0,-9.642109e+05,0.0,4.316303e+06,4.308138e+06,0.036366,1.057178
4,2021-03-31,5741000.0,-1.506210e+06,0.0,5.927135e+06,5.916303e+06,0.132692,1.060488


In [6]:
funds.head()

,investment_name,quarter_end,par_value,invested_amount,base_rate,sofr_rate,rate_status,spread,pik_rate,cash_interest_rate,...,prepayment_date,beginning_balance,amortization,pik_interest,ending_balance,cash_interest,fees,remaining_balance_payment,total_payment,irr
0,A Corp Term Loan,2020-03-31,1000000,970000.0,SOFR,0.0001,actual,0.00,0.00,0.0000,...,None,1.000000e+06,0.0,0.000000,1.000000e+06,0.000000,0.0,0.0,-970000.000000,NaN
1,A Corp Term Loan,2020-06-30,1000000,970000.0,SOFR,0.0010,actual,0.07,0.02,0.0510,...,None,1.000000e+06,2500.0,5000.000000,1.002500e+06,12750.000000,0.0,0.0,15250.000000,0.024506
2,A Corp Term Loan,2020-09-30,1000000,970000.0,SOFR,0.0008,actual,0.07,0.02,0.0508,...,None,1.002500e+06,2500.0,5012.500000,1.005012e+06,12731.750000,0.0,0.0,15231.750000,0.022357
3,A Corp Term Loan,2020-12-31,1000000,970000.0,SOFR,0.0007,actual,0.07,0.02,0.0507,...,None,1.005012e+06,2500.0,5025.062500,1.007538e+06,12738.533438,0.0,0.0,15238.533438,0.087868
4,A Corp Term Loan,2021-03-31,1000000,970000.0,SOFR,0.0001,actual,0.07,0.02,0.0501,...,None,1.007538e+06,2500.0,5037.687813,1.010075e+06,12619.407970,0.0,0.0,15119.407970,0.084986


In [7]:
funds_summary

,investment_name,investment_date,maturity_date,par_value,spread,base_rate,sofr_assumption,pik_interest,amortization,oid,exit_fee,prepayment_date,total_payment,irr
0,A Corp Term Loan,2020-03-31,2025-06-30,1000000,0.07,SOFR,actual,0.02,0.01,0.03,0.02,None,5.614351e+05,0.102485
1,B Corp Term Loan,2020-06-30,2027-06-30,1250000,0.06,SOFR,actual,0.02,0.01,0.02,0.02,None,8.610207e+05,0.093598
2,C Corp Term Loan,2020-09-30,2026-12-31,1000000,0.06,SOFR,actual,0.02,0.01,0.03,0.02,None,6.302986e+05,0.096815
3,D Corp Term Loan,2020-12-31,2028-03-31,1050000,0.08,SOFR,actual,0.02,0.01,0.04,0.02,None,9.485280e+05,0.121995
4,E Corp Term Loan,2021-03-31,2029-06-30,1600000,0.07,SOFR,actual,0.02,0.01,0.02,0.02,None,1.492826e+06,0.109149
